In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd

from utils.indicators import *
from utils.engine import *

import lightgbm as lgb
from sklearn.metrics import precision_score, recall_score, classification_report
import matplotlib.pyplot as plt

# ============================================================================
# CONFIG
# ============================================================================
LOOKAHEAD_SECONDS   = 240     # how far forward we check if the barrier was hit
BARRIER_MULTIPLIER  = 1.0     # profit/stop barrier = multiplier * horizon-scaled vol
RSI_FILTER_COLUMN   = 'rsi_2'
RSI_FILTER_THRESHOLD = 30
EPS                 = 1e-9    # guards against division by zero

# --- Chronological Validation Settings ---
N_RANDOM_SAMPLES    = 15      # Number of sequential walk-forward folds to test
TEST_BLOCK_SIZE     = 10000   # Number of rows per test chunk
EMBARGO_SECONDS     = LOOKAHEAD_SECONDS

# --- 3-Class Label Encoding ---
# WIN     = profit barrier was reached AND the stop barrier was never breached
# LOSS    = stop barrier was breached (regardless of whether price also
#           technically touched the profit barrier somewhere in the window —
#           this mirrors the original binary logic, which also called that
#           case a "0" since it can't tell barrier-touch order from mfe/mae alone)
# TIMEOUT = neither barrier was breached before the lookahead window ran out
LABEL_LOSS    = 0
LABEL_TIMEOUT = 1
LABEL_WIN     = 2
CLASS_NAMES   = {LABEL_LOSS: 'LOSS', LABEL_TIMEOUT: 'TIMEOUT', LABEL_WIN: 'WIN'}


# ============================================================================
# STEP 1 — Per-session feature pipeline
# ============================================================================
def load_and_build_state_space(orderbook_path, trades_path, session_id):
    df_orderbook = pd.read_csv(orderbook_path)
    df_trades = pd.read_csv(trades_path)

    df_orderbook['datetime'] = pd.to_datetime(df_orderbook['timestamp'].astype('int64') // 1000, unit='s')
    df_trades['datetime'] = pd.to_datetime(df_trades['timestamp'].astype('int64') // 1000, unit='s')

    df_orderbook = df_orderbook.set_index('datetime').sort_index()
    df_trades = df_trades.set_index('datetime').sort_index()

    df_orderbook['mid_price'] = (df_orderbook['bid_price'] + df_orderbook['ask_price']) / 2

    df_ob_1m = df_orderbook.resample('1min').agg({
        'mid_price': ['first', 'max', 'min', 'last'],
        'bid_price': 'last',
        'ask_price': 'last',
        'bid_amount': 'sum',
        'ask_amount': 'sum',
    })
    df_ob_1m.columns = ['open', 'high', 'low', 'close', 'bid_price', 'ask_price', 'bid_amount', 'ask_amount']
    df_ob_1m['spread'] = df_ob_1m['ask_price'] - df_ob_1m['bid_price']

    df_trades['buy_vol'] = df_trades['amount'].where(df_trades['side'] == 'buy', 0)
    df_trades['sell_vol'] = df_trades['amount'].where(df_trades['side'] == 'sell', 0)

    df_micro = df_trades.resample('1s').agg({
        'price': 'last',
        'buy_vol': 'sum',
        'sell_vol': 'sum'
    }).ffill()

    df_micro['micro_OFI'] = df_micro['buy_vol'] - df_micro['sell_vol']
    df_micro['micro_CVD'] = df_micro['micro_OFI'].rolling(10).sum()

    df_trades_1m = df_trades.resample('1min').agg({
        'price': ['first', 'max', 'min', 'last', 'count'],
        'amount': 'sum',
        'buy_vol': ['sum', 'max'],
        'sell_vol': ['sum', 'max']
    })
    df_trades_1m.columns = ['open', 'high', 'low', 'close', 'num_trades', 'volume',
                             'agg_buyer', 'largest_buyer', 'agg_seller', 'largest_seller']
    df_trades_1m = df_trades_1m.dropna(subset=['close'])

    df_price_1m = pd.DataFrame({
        'open': df_ob_1m['open'],
        'high': df_ob_1m['high'],
        'low': df_ob_1m['low'],
        'close': df_ob_1m['close'],
        'largest_buyer': df_trades_1m['largest_buyer'] * 100 / (df_trades_1m['volume'] + EPS),
        'largest_seller': df_trades_1m['largest_seller'] * 100 / (df_trades_1m['volume'] + EPS),
        'OFI_LOB': 100 * (df_ob_1m['bid_amount'] - df_ob_1m['ask_amount']) / (df_ob_1m['bid_amount'] + df_ob_1m['ask_amount'] + EPS),
        'OFI_AGG': 100 * (df_trades_1m['agg_buyer'] - df_trades_1m['agg_seller']) / (df_trades_1m['agg_buyer'] + df_trades_1m['agg_seller'] + EPS),
        'CVD_raw': calculate_cvd(df_trades_1m['agg_buyer'], df_trades_1m['agg_seller']),
        'candle_type': abs(df_ob_1m['close'] - df_ob_1m['open']) / (abs(df_ob_1m['high'] - df_ob_1m['low']) + EPS),
        'candle_size': (df_ob_1m['high'] - df_ob_1m['low']) / ((df_ob_1m['high'] - df_ob_1m['low']).rolling(14).mean() + EPS),
        'volume': df_trades_1m['volume']
    })

    cvd_delta = df_price_1m['CVD_raw'].diff()
    df_price_1m['CVD_z'] = (cvd_delta - cvd_delta.rolling(60).mean()) / (cvd_delta.rolling(60).std() + EPS)
    df_price_1m = df_price_1m.drop(columns=['CVD_raw'])

    price_clean = np.asarray(df_price_1m['close'], dtype=float)
    diffs = np.diff(price_clean)

    def _pad_front(arr, target_len):
        out = np.full(target_len, np.nan)
        out[-len(arr):] = arr
        return out

    df_price_1m['rsi_2'] = _pad_front(get_wilders_rsi(diffs, 2), len(df_price_1m))
    rsi_14_raw = get_wilders_rsi(diffs, 14)
    df_price_1m['rsi_14'] = _pad_front(rsi_14_raw, len(df_price_1m))
    df_price_1m['sma_14'] = _pad_front(sma(rsi_14_raw, 14), len(df_price_1m))

    df_micro_shifted = df_micro.shift(1)
    df_macro_shifted = df_price_1m

    df_state_space = df_micro_shifted.join(df_macro_shifted, how='left').ffill()
    
    # NEW: Keep the true, unlagged price for trade execution/labeling
    df_state_space['execution_price'] = df_micro['price'] 
    
    df_state_space = df_state_space.dropna()
    df_state_space['session_id'] = session_id

    return df_state_space


# ============================================================================
# STEP 2 — Build combined state space from all sessions
# ============================================================================

session_0 = load_and_build_state_space(
    "./csv/BTCUSDT_orderbook.csv", "./csv/BTCUSDT_trades.csv", session_id=0
)
session_1 = load_and_build_state_space(
    "./csv/BTCUSDT_orderbook_2.csv", "./csv/BTCUSDT_trades_2.csv", session_id=1
)
session_2 = load_and_build_state_space(
    "./csv/BTCUSDT_orderbook_3.csv", "./csv/BTCUSDT_trades_3.csv", session_id=1
)

#df_state_space = load_and_build_state_space("./csv/XAUTUSDT_orderbook.csv", "./csv/XAUTUSDT_trades.csv", 0)

# Approach 1: Triple Class Classifier
Three classes are defined.
- Timeout: If the trade exits out of timeout.
- Loss: If the trade hits stop barrier.
- Win: If the trade hits profit barrier.

In [4]:

# ============================================================================
# STEP 3 — Barrier labeling (Preserving paths for exact backtesting)
# ============================================================================
def add_barrier_labels(df, lookahead_seconds=LOOKAHEAD_SECONDS, barrier_mult=BARRIER_MULTIPLIER):
    labeled = []
    for _, g in df.groupby('session_id', sort=False):
        g = g.copy()
        ret_std_1s = g['price'].pct_change().rolling(window=100).std()
        horizon_std = ret_std_1s * np.sqrt(lookahead_seconds)

        g['profit_barrier'] = horizon_std * barrier_mult
        g['stop_barrier'] = horizon_std * barrier_mult

        forward_max = (g['price'].iloc[::-1]
                       .rolling(window=lookahead_seconds, min_periods=lookahead_seconds)
                       .max().iloc[::-1])
        forward_min = (g['price'].iloc[::-1]
                       .rolling(window=lookahead_seconds, min_periods=lookahead_seconds)
                       .min().iloc[::-1])

        # Added to capture absolute path deviations & terminal returns accurately
        forward_close = g['price'].shift(-lookahead_seconds)

        g['mfe'] = (forward_max - g['price']) / g['price']
        g['mae'] = (g['price'] - forward_min) / g['price']
        g['terminal_ret'] = (forward_close - g['price']) / g['price']

        valid = g['profit_barrier'].notna() & forward_max.notna() & forward_min.notna()

        # Same underlying comparisons as before, just split into three
        # mutually-exclusive outcomes instead of collapsing "loss" and
        # "timeout" together into a single 0 class.
        is_win = (g['mfe'] >= g['profit_barrier']) & (g['mae'] < g['stop_barrier'])
        is_loss = (g['mae'] >= g['stop_barrier'])
        # anything left over (neither barrier breached inside the window) is a timeout

        g['target'] = np.nan
        g.loc[valid, 'target'] = np.select(
            [is_win[valid], is_loss[valid]],
            [LABEL_WIN, LABEL_LOSS],
            default=LABEL_TIMEOUT
        )
        labeled.append(g)

    out = pd.concat(labeled)
    out = out.dropna(subset=['target'])
    out['target'] = out['target'].astype(int)
    return out

df_state_space = add_barrier_labels(df_state_space)

label_counts = df_state_space['target'].map(CLASS_NAMES).value_counts()
print(f"Label distribution (full dataset): {label_counts.to_dict()}")

df_training_zone = df_state_space[df_state_space[RSI_FILTER_COLUMN] < RSI_FILTER_THRESHOLD]
print(f"Total rows in hunting zone: {len(df_training_zone)}")
print(f"Label distribution (hunting zone): {df_training_zone['target'].map(CLASS_NAMES).value_counts().to_dict()}")


# ============================================================================
# STEP 4 — Features / target configuration
# ============================================================================
# FIXED: Explicitly added backtest telemetry path variables into the training drop list
drop_cols = ['target', 'price', 'open', 'high', 'low', 'close',
             'profit_barrier', 'stop_barrier', 'session_id', 'mfe', 'mae', 'terminal_ret']
features = [c for c in df_training_zone.columns if c not in drop_cols]

X = df_training_zone[features]
y = df_training_zone['target']


# ============================================================================
# STEP 5 — Purged Chronological Walk-Forward Splitting
# ============================================================================
def purged_chronological_splits(index, n_samples, block_size, horizon_seconds, embargo_seconds=0):
    n = len(index)
    horizon = pd.Timedelta(seconds=horizon_seconds)
    min_initial_train = int(n * 0.35)
    available_test_run = n - min_initial_train - block_size

    if available_test_run <= 0:
        raise ValueError("The dataset length is too narrow for this test configuration block size.")

    step_intervals = available_test_run // n_samples

    for fold in range(n_samples):
        start_idx = min_initial_train + (fold * step_intervals)
        test_idx = np.arange(start_idx, start_idx + block_size)
        test_start_time = index[test_idx[0]]

        possible_train_idx = np.arange(0, start_idx)
        train_times = index[possible_train_idx]

        purge_mask = (train_times + horizon >= test_start_time)
        train_idx = possible_train_idx[~purge_mask]

        n_purged = len(possible_train_idx) - len(train_idx)
        yield train_idx, test_idx, n_purged

splitter = purged_chronological_splits(
    X.index,
    n_samples=N_RANDOM_SAMPLES,
    block_size=TEST_BLOCK_SIZE,
    horizon_seconds=LOOKAHEAD_SECONDS,
    embargo_seconds=EMBARGO_SECONDS
)

print(f"\n--- Starting {N_RANDOM_SAMPLES} Chronological Walk-Forward Validation Folds (Size: {TEST_BLOCK_SIZE}) ---")

thresholds_to_track = np.arange(0.50, 0.96, 0.05)
fold_metrics = {t: {'precisions': [], 'trades': []} for t in thresholds_to_track}
fold_importances = []

for fold, (train_idx, test_idx, n_purged) in enumerate(splitter, start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    if len(X_train) == 0 or y_train.nunique() < 2:
        print(f"Fold {fold} | skipped — invalid training data distribution.")
        continue

    # scale_pos_weight is a binary-only LightGBM knob, so imbalance across the
    # three classes (LOSS / TIMEOUT / WIN) is handled with class_weight='balanced'
    # instead — the multiclass equivalent of the original neg_pos_ratio reweighting.
    clf = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        num_leaves=31,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    # Before training your LightGBM model:
    features_for_training = df_state_space.drop(columns=['execution_price', 'session_id'])

    clf.fit(X_train, y_train)

    if LABEL_WIN not in clf.classes_:
        print(f"Fold {fold} | skipped scoring — no WIN samples present in this fold's training data.")
        continue

    # predict_proba now returns one column per class present in this fold;
    # we pull out the WIN column, which is the direct multiclass analogue of
    # the old binary predict_proba(...)[:, 1] "probability of a winning trade".
    win_idx = list(clf.classes_).index(LABEL_WIN)
    y_proba = clf.predict_proba(X_test)[:, win_idx]
    y_test_is_win = (y_test == LABEL_WIN).astype(int)

    for t in thresholds_to_track:
        preds = (y_proba >= t).astype(int)
        prec = precision_score(y_test_is_win, preds, zero_division=0)
        num_trades = int(preds.sum())

        if num_trades > 0:
            fold_metrics[t]['precisions'].append(prec)
        fold_metrics[t]['trades'].append(num_trades)

    fold_importances.append(clf.feature_importances_)
    print(f"Fold {fold:02d} | Train Rows: {len(X_train):<6} | Tested on forward {TEST_BLOCK_SIZE} rows. Purged: {n_purged}")


# ============================================================================
# STEP 6 — Threshold Fluctuation Reporting
# ============================================================================
print("\n--- Edge Stability Across Chronological Windows (Honest Analysis, WIN class) ---")
print("-" * 75)
print(f"{'Threshold':<10} | {'Mean Prec':<11} | {'Std Dev (Fluct)':<16} | {'Avg Trades per Block'}")
print("-" * 75)

for t in thresholds_to_track:
    prec_array = np.array(fold_metrics[t]['precisions'])
    trades_array = np.array(fold_metrics[t]['trades'])

    if len(prec_array) > 0:
        mean_prec = np.mean(prec_array)
        std_prec = np.std(prec_array)
        mean_trades = np.mean(trades_array)

        print(f">= {t*100:>4.1f}%   | {mean_prec:>9.2%}   | ± {std_prec:<13.2%} | {mean_trades:>6.1f}")
    else:
        print(f">= {t*100:>4.1f}%   | No trades fired in any test block.")

print("-" * 75)

if fold_importances:
    imp_df = pd.DataFrame(fold_importances, columns=features)
    imp_summary = pd.DataFrame({
        'mean_importance': imp_df.mean(),
        'std_importance': imp_df.std()
    }).sort_values('mean_importance', ascending=False)

    print("\n--- Top 10 Feature Importance Stability ---")
    print(imp_summary.head(10))

Label distribution (full dataset): {'TIMEOUT': 166338, 'LOSS': 88694, 'WIN': 26101}
Total rows in hunting zone: 223673
Label distribution (hunting zone): {'TIMEOUT': 146215, 'LOSS': 68375, 'WIN': 9083}

--- Starting 15 Chronological Walk-Forward Validation Folds (Size: 10000) ---
Fold 01 | Train Rows: 78165  | Tested on forward 10000 rows. Purged: 120
Fold 02 | Train Rows: 87070  | Tested on forward 10000 rows. Purged: 240
Fold 03 | Train Rows: 96095  | Tested on forward 10000 rows. Purged: 240
Fold 04 | Train Rows: 105120 | Tested on forward 10000 rows. Purged: 240
Fold 05 | Train Rows: 114145 | Tested on forward 10000 rows. Purged: 240
Fold 06 | Train Rows: 123170 | Tested on forward 10000 rows. Purged: 240
Fold 07 | Train Rows: 132195 | Tested on forward 10000 rows. Purged: 240
Fold 08 | Train Rows: 141220 | Tested on forward 10000 rows. Purged: 240
Fold 09 | Train Rows: 150245 | Tested on forward 10000 rows. Purged: 240
Fold 10 | Train Rows: 159270 | Tested on forward 10000 rows. P

# Patch for ml-state code by AI
Integrate random splitting of data to ensure the code truly works and there is no overfitting problem. How we test the stability of machine learning algo:
- Run on large-sample sized splits (20k). Measure Mean Precision of trades across sample size then get Std Dev to get flucutation of performance in machine learning algo.
- Deduce based on expectancy formula if strtaegy is profitable or not. Further validation by changing sample size, switching tickers, etc.

# Conclusion (20k samples)
![performance](./image/img_12.png)

With condition 2RR, trading on 85%-95% shows a very stable edge.
- Average no of trades: 5.67 trades per hour
- 85.0%: 52.36% * 2RR - 47.64% = 0.5708RR
- Worst condition: (52.36% - 22.98%) * 2RR - (100% - 52.36% + 22.98%) = 0.5876 - 0.7062 = -0.1186RR
- Using assumption that it is a normal distribution, 65% of the time, the model performs reasonably well and generally has a positive expectancy. However there is a 5% chance that the model experiences severe underperformance (2x stdev)

# Conclusion (10k samples split)
![Performance](./image/img_13.png)

With condition 2RR, trading on 85%-95% shows a very stable edge.
- Average no of trades: 3.348 trades per hour
- 85.0%: 62.37% * 2RR - 37.63% = 0.8711RR
- Worst condition: (62.37% - 21.44%) * 2RR - (100% - 62.37% + 21.44%) = 0.8186 - 0.5907 = 0.2279RR
- Using assumption that it is a normal distribution, 65% of the time, the model performs exceptionally well and generally has a positive expectancy even when the market is 1stdev worse. For the strategy to hit breakeven, it should hit the 1.38x stdev area.

Nevermind ignore this conclusion for now. realized lookahead bias

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

from utils.indicators import *
from utils.engine import *

import lightgbm as lgb
from sklearn.metrics import precision_score, recall_score, classification_report
import matplotlib.pyplot as plt
import shap
from scipy import stats
from sklearn.inspection import permutation_importance

# ============================================================================
# CONFIG
# ============================================================================
LOOKAHEAD_SECONDS    = 300     # how far forward we walk the price path looking for a barrier touch
BARRIER_MULTIPLIER   = 2.0     # profit/stop barrier = multiplier * horizon-scaled vol
RSI_FILTER_COLUMN    = 'rsi_2'
RSI_FILTER_THRESHOLD = 30
EPS                  = 1e-9    # guards against division by zero
BARRIER_CHUNK_SIZE   = 50_000  # rows processed per chunk in the vectorized barrier walk (memory control)

# --- Chronological Validation Settings ---
N_RANDOM_SAMPLES    = 15      # Number of sequential walk-forward folds to test
TEST_BLOCK_SIZE     = 10000   # Number of rows per test chunk
EMBARGO_SECONDS      = LOOKAHEAD_SECONDS

# --- 2-Class Binary Label Encoding ---
# WIN  = Profit barrier touched FIRST, OR timeout occurred with price > entry
# LOSS = Stop barrier touched FIRST, OR timeout occurred with price <= entry
LABEL_LOSS    = 0
LABEL_WIN     = 1
CLASS_NAMES   = {LABEL_LOSS: 'LOSS', LABEL_WIN: 'WIN'}
RR_RATIO      = 1.0

def _sequential_triple_barrier(price, profit_thresh, stop_thresh, lookahead, chunk_size=BARRIER_CHUNK_SIZE):
    price = np.asarray(price, dtype=float)
    profit_thresh = np.asarray(profit_thresh, dtype=float)
    stop_thresh = np.asarray(stop_thresh, dtype=float)
    n = len(price)

    label = np.full(n, LABEL_LOSS, dtype=int)
    exit_offset = np.full(n, -1, dtype=int)
    exit_ret = np.full(n, np.nan)
    terminal_ret = np.full(n, np.nan)
    is_timeout = np.zeros(n, dtype=bool)
    valid = np.zeros(n, dtype=bool)

    usable_n = n - lookahead 
    if usable_n <= 0:
        return label, exit_offset, exit_ret, terminal_ret, is_timeout, valid

    for c0 in range(0, usable_n, chunk_size):
        c1 = min(c0 + chunk_size, usable_n)

        p0 = price[c0:c1]
        pt = profit_thresh[c0:c1]
        st = stop_thresh[c0:c1]

        upper = p0 * (1.0 + (pt * RR_RATIO))
        lower = p0 * (1.0 - st)

        window = sliding_window_view(price[c0 + 1: c1 + lookahead], lookahead) 

        hit_upper = window >= upper[:, None]
        hit_lower = window <= lower[:, None]

        has_upper = hit_upper.any(axis=1)
        has_lower = hit_lower.any(axis=1)
        first_upper = np.argmax(hit_upper, axis=1) 
        first_lower = np.argmax(hit_lower, axis=1) 

        win_mask = has_upper & (~has_lower | (first_upper < first_lower))
        loss_mask = has_lower & (~has_upper | (first_lower < first_upper))
        
        timeout_mask = ~(win_mask | loss_mask)
        timeout_win_mask = timeout_mask & (window[:, -1] > p0)
        timeout_loss_mask = timeout_mask & (window[:, -1] <= p0)

        thresholds_ok = ~np.isnan(pt) & ~np.isnan(st)

        lab = np.full(c1 - c0, LABEL_LOSS, dtype=int)
        lab[win_mask | timeout_win_mask] = LABEL_WIN
        lab[loss_mask | timeout_loss_mask] = LABEL_LOSS

        off = np.full(c1 - c0, lookahead - 1, dtype=int) 
        off[win_mask] = first_upper[win_mask]
        off[loss_mask] = first_lower[loss_mask]

        exit_price = window[np.arange(c1 - c0), off]

        label[c0:c1] = lab
        exit_offset[c0:c1] = off
        exit_ret[c0:c1] = (exit_price - p0) / p0
        terminal_ret[c0:c1] = (window[:, -1] - p0) / p0
        is_timeout[c0:c1] = timeout_mask
        valid[c0:c1] = thresholds_ok

    return label, exit_offset, exit_ret, terminal_ret, is_timeout, valid


def add_barrier_labels(df, lookahead_seconds=LOOKAHEAD_SECONDS, barrier_mult=BARRIER_MULTIPLIER, timeframe=60):
    labeled = []
    for _, g in df.groupby('session_id', sort=False):
        g = g.copy()
        
        # Calculate volatility on the continuous execution price
        ret_std_1s = g['execution_price'].pct_change().rolling(window=100).std()
        horizon_std = ret_std_1s * np.sqrt(timeframe)

        g['profit_barrier'] = horizon_std * barrier_mult
        g['stop_barrier'] = horizon_std * barrier_mult

        # Pass the unlagged execution_price to the labeler
        label, exit_offset, exit_ret, terminal_ret, is_timeout, valid = _sequential_triple_barrier(
            g['execution_price'].values, g['profit_barrier'].values, g['stop_barrier'].values, lookahead_seconds
        )
        
        g['target'] = label
        g['bars_to_exit'] = exit_offset      
        g['exit_ret'] = exit_ret             
        g['terminal_ret'] = terminal_ret     
        g['is_timeout'] = is_timeout
        g['_valid'] = valid

        g = g[g['_valid']].drop(columns=['_valid'])
        g['target'] = g['target'].astype(int)

        labeled.append(g)

    return pd.concat(labeled)

df_state_space = pd.concat([session_0, session_1, session_2]).sort_index()
df_state_space = add_barrier_labels(df_state_space)

df_state_space = df_state_space[df_state_space['volume'] > 0] 
df_training_zone = df_state_space[df_state_space[RSI_FILTER_COLUMN] < RSI_FILTER_THRESHOLD]

label_counts = df_state_space['target'].map(CLASS_NAMES).value_counts()
print(f"Label distribution (full dataset): {label_counts.to_dict()}")

df_training_zone = df_state_space[df_state_space[RSI_FILTER_COLUMN] < RSI_FILTER_THRESHOLD]
print(f"Total rows in hunting zone: {len(df_training_zone)}")
print(f"Label distribution (hunting zone): {df_training_zone['target'].map(CLASS_NAMES).value_counts().to_dict()}")


# ============================================================================
# STEP 4 — Features / target configuration
# ============================================================================
drop_cols = ['target', 'price', 'open', 'high', 'low', 'close',
             'profit_barrier', 'stop_barrier', 'session_id',
             'bars_to_exit', 'exit_ret', 'terminal_ret', 'is_timeout']
features = [c for c in df_training_zone.columns if c not in drop_cols]

X = df_training_zone[features]
y = df_training_zone['target']

# ============================================================================
# STEP 5 — Risk-Normalized Return (R-Multiple Bookkeeping)
# ============================================================================
normalized_return = pd.Series(index=df_training_zone.index, dtype=float)

is_timeout = df_training_zone['is_timeout']
is_win = df_training_zone['target'] == LABEL_WIN
is_loss = df_training_zone['target'] == LABEL_LOSS

normalized_return[is_win & ~is_timeout] = RR_RATIO
normalized_return[is_loss & ~is_timeout] = -1.0

normalized_return[is_timeout] = (
    df_training_zone.loc[is_timeout, 'exit_ret'] /
    (df_training_zone.loc[is_timeout, 'stop_barrier'] + EPS)
)

# --- USER PLACEMENT: Calculate the average absolute stop barrier size ---
mean_stop_barrier = df_training_zone['stop_barrier'].mean()

print("\n" + "="*50)
print("STOP BARRIER METRICS")
print("="*50)
print(f"Average Stop Barrier Size (Return): {mean_stop_barrier:.6f}")
print(f"Average Stop Barrier Size (%):      {mean_stop_barrier * 100:.4f}%")
print(f"Min Stop Barrier Size (%):          {df_training_zone['stop_barrier'].min() * 100:.4f}%")
print(f"Max Stop Barrier Size (%):          {df_training_zone['stop_barrier'].max() * 100:.4f}%")
print("="*50)


# ============================================================================
# STEP 6 — Purged Chronological Walk-Forward Splitting
# ============================================================================
TARGET_TRADE_CLASS = LABEL_WIN
INVERT_RETURNS     = True      

target_name = CLASS_NAMES[TARGET_TRADE_CLASS]
prob_label = f"P({target_name})"

def purged_chronological_splits(index, n_samples, block_size, horizon_seconds, embargo_seconds=0):
    n = len(index)
    horizon = pd.Timedelta(seconds=horizon_seconds)
    embargo = pd.Timedelta(seconds=embargo_seconds)
    min_initial_train = int(n * 0.35)
    available_test_run = n - min_initial_train - block_size

    if available_test_run <= 0:
        raise ValueError("The dataset length is too narrow for this test configuration block size.")

    step_intervals = available_test_run // n_samples

    for fold in range(n_samples):
        start_idx = min_initial_train + (fold * step_intervals)
        test_idx = np.arange(start_idx, start_idx + block_size)
        test_start_time = index[test_idx[0]]

        possible_train_idx = np.arange(0, start_idx)
        train_times = index[possible_train_idx]

        purge_mask = (train_times + horizon + embargo >= test_start_time)
        train_idx = possible_train_idx[~purge_mask]

        n_purged = len(possible_train_idx) - len(train_idx)
        yield train_idx, test_idx, n_purged

splitter = purged_chronological_splits(
    X.index,
    n_samples=N_RANDOM_SAMPLES,
    block_size=TEST_BLOCK_SIZE,
    horizon_seconds=LOOKAHEAD_SECONDS,
    embargo_seconds=EMBARGO_SECONDS
)

print(f"\n--- Starting {N_RANDOM_SAMPLES} Walk-Forward Folds | Targeting: {target_name} | Inverting Trades: {INVERT_RETURNS} ---")

thresholds_to_track = np.arange(0.50, 0.96, 0.05)
fold_metrics = {t: {'precisions': [], 'win_rates': [], 'rrs': [], 'trades': []} for t in thresholds_to_track}
fold_importances = []
trade_log = {t: [] for t in thresholds_to_track}       
fold_detail_records = []                                      

class_labels_ordered = [LABEL_LOSS, LABEL_WIN]
class_metrics_by_label = {lbl: {'precision': [], 'recall': []} for lbl in class_labels_ordered}
class_metrics_records = []
all_y_test_pooled = []
all_y_pred_pooled = []

fold_shap_values = []
fold_pfi_values = []  # <--- ADD THIS LINE

for fold, (train_idx, test_idx, n_purged) in enumerate(splitter, start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    if len(X_train) == 0 or y_train.nunique() < 2:
        print(f"Fold {fold} | skipped — invalid training data distribution.")
        continue

    clf = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=31,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    clf.fit(X_train, y_train)

    # ---------------------------------------------------------
    # Safely calculate SHAP
    # ---------------------------------------------------------
    try:
        X_test_sampled = X_test.sample(min(1000, len(X_test)), random_state=42)
        explainer = shap.TreeExplainer(clf)
        shap_vals = explainer.shap_values(X_test_sampled)
        
        # Handle different SHAP versions and output formats
        if isinstance(shap_vals, list):
            # Older SHAP version (list of arrays, index 1 is positive class)
            shap_array = shap_vals[1]
        elif hasattr(shap_vals, 'values'):
            # Newer SHAP version (Explanation object)
            if len(shap_vals.values.shape) == 3:
                shap_array = shap_vals.values[:, :, 1]
            else:
                shap_array = shap_vals.values
        else:
            # Fallback for standard 2D numpy array return
            shap_array = shap_vals
            
        # Calculate mean ABSOLUTE impact per feature for this fold
        fold_mean_shap = np.abs(shap_array).mean(axis=0)
        fold_shap_values.append(fold_mean_shap)
        
    except Exception as e:
        print(f"Fold {fold:02d} | SHAP calculation failed: {e}")
        fold_shap_values.append(np.full(len(features), np.nan))
    
    # ... existing SHAP code ...
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
        
    if y_test.nunique() < 2:
        print(f"Fold {fold:02d} | Skipped PFI — y_test lacks both classes (cannot compute ROC-AUC).")
        fold_pfi_values.append(np.full(len(features), np.nan))
    else:
        try:
            pi_result = permutation_importance(
                clf, X_test, y_test,
                n_repeats=5,
                random_state=42,
                scoring='roc_auc',
                n_jobs=1  # Prevent thread contention with LightGBM
            )
            fold_pfi_values.append(pi_result.importances_mean)
        except Exception as e:
            print(f"Fold {fold:02d} | PFI calculation failed: {e}")
            fold_pfi_values.append(np.full(len(features), np.nan))

    y_pred_argmax = clf.predict(X_test)
    prec_per_class = precision_score(y_test, y_pred_argmax, labels=class_labels_ordered, average=None, zero_division=0)
    rec_per_class = recall_score(y_test, y_pred_argmax, labels=class_labels_ordered, average=None, zero_division=0)

    for lbl, p, r in zip(class_labels_ordered, prec_per_class, rec_per_class):
        class_metrics_by_label[lbl]['precision'].append(p)
        class_metrics_by_label[lbl]['recall'].append(r)
        class_metrics_records.append({
            'fold': fold,
            'class': CLASS_NAMES[lbl],
            'precision': p,
            'recall': r,
            'support': int((y_test == lbl).sum()),
        })

    all_y_test_pooled.append(y_test.values)
    all_y_pred_pooled.append(y_pred_argmax)

    if TARGET_TRADE_CLASS not in clf.classes_:
        print(f"Fold {fold} | skipped threshold scoring — no {target_name} samples in training data.")
        continue

    target_idx = list(clf.classes_).index(TARGET_TRADE_CLASS)
    y_proba = clf.predict_proba(X_test)[:, target_idx]
    
    for t in thresholds_to_track:
        preds = (y_proba >= t).astype(int)
        
        # --- FIXED: Chronological Non-Overlapping Position Filter ---
        signal_indices = np.where(preds == 1)[0]
        non_overlapping_idx = []
        next_available_time = pd.Timestamp.min  # Track actual time, not row index

        for idx in signal_indices:
            actual_global_time = X_test.index[idx]
            
            # If this signal happens while we are already in a trade, skip it
            if actual_global_time < next_available_time:
                continue 
                
            non_overlapping_idx.append(actual_global_time)
            
            # Get the duration in seconds
            trade_duration_seconds = int(df_training_zone.loc[actual_global_time, 'bars_to_exit']) + 1
            
            # The next time we can trade is the current time + the trade duration
            next_available_time = actual_global_time + pd.Timedelta(seconds=trade_duration_seconds)

        num_trades = len(non_overlapping_idx)

        if num_trades > 0:
            traded_idx = pd.Index(non_overlapping_idx)
            traded_targets = y_test.loc[traded_idx]
            
            traded_returns = normalized_return.loc[traded_idx].copy()
            if INVERT_RETURNS:
                traded_returns = -traded_returns

            trade_log[t].extend(zip(traded_idx, traded_returns.values))

            num_original_wins = int((traded_targets == LABEL_WIN).sum())
            num_original_losses = int((traded_targets == LABEL_LOSS).sum())

            strategy_wins = num_original_losses if INVERT_RETURNS else num_original_wins
            strategy_losses = num_original_wins if INVERT_RETURNS else num_original_losses

            win_rate = strategy_wins / num_trades

            avg_win_ret = traded_returns[traded_returns > 0].mean() if strategy_wins > 0 else np.nan
            avg_loss_ret = traded_returns[traded_returns < 0].mean() if strategy_losses > 0 else np.nan
            rr = (avg_win_ret / abs(avg_loss_ret)) if (not np.isnan(avg_win_ret) and not np.isnan(avg_loss_ret) and avg_loss_ret != 0) else np.nan

            # Precision recalculated dynamically for the authentic subset
            prec = precision_score(y_test.loc[traded_idx] == TARGET_TRADE_CLASS, np.ones(num_trades), zero_division=0)

            fold_metrics[t]['precisions'].append(prec)
            fold_metrics[t]['win_rates'].append(win_rate)
            fold_metrics[t]['rrs'].append(rr)

            fold_detail_records.append({
                'threshold': round(float(t), 2),
                'fold': fold,
                'trades': num_trades,
                'strategy_wins': strategy_wins,
                'strategy_losses': strategy_losses,
                'precision': prec,
                'win_rate': win_rate,
                'rr': rr,
                'avg_return': traded_returns.mean(),
            })

        fold_metrics[t]['trades'].append(num_trades)

    fold_importances.append(clf.feature_importances_)
    print(f"Fold {fold:02d} | Train Rows: {len(X_train):<6} | Tested on forward {TEST_BLOCK_SIZE} rows. Purged: {n_purged}")

# ============================================================================
# STEP 8 — SHAP vs PERMUTATION IMPORTANCE
# ============================================================================

print("\n--- Feature Importance: SHAP vs Permutation (PFI) ---")
print("-" * 80)

# Average the values across all folds, explicitly ignoring NaNs
mean_shap = np.mean(fold_shap_values, axis=0)
mean_pfi = np.nanmean(fold_pfi_values, axis=0)  

importance_df = pd.DataFrame({
    'Feature': features,
    'SHAP_Importance': mean_shap,
    'PFI_Mean_AUC_Drop': mean_pfi
}).sort_values('PFI_Mean_AUC_Drop', ascending=False)

# ADD THESE LINES TO ACTUALLY PRINT THE DATAFRAME
print(importance_df.to_string(index=False))

# Optional: Save it to a CSV so you don't lose the data when the console clears
importance_df.to_csv('feature_importance.csv', index=False)

# ============================================================================
# STEP 9 — Model Class-Level Precision & Recall (argmax predictions)
# ============================================================================

print("\n--- Model Binary Precision & Recall (argmax prediction) — Mean ± Std Across Folds ---")
print("-" * 70)
print(f"{'Class':<10} | {'Precision':<20} | {'Recall'}")
print("-" * 70)
for lbl in class_labels_ordered:
    p_arr = np.array(class_metrics_by_label[lbl]['precision'])
    r_arr = np.array(class_metrics_by_label[lbl]['recall'])
    p_txt = f"{p_arr.mean():>6.2%} ± {p_arr.std():<6.2%}" if len(p_arr) else "n/a"
    r_txt = f"{r_arr.mean():>6.2%} ± {r_arr.std():<6.2%}" if len(r_arr) else "n/a"
    print(f"{CLASS_NAMES[lbl]:<10} | {p_txt:<20} | {r_txt}")
print("-" * 70)

if class_metrics_records:
    class_metrics_df = pd.DataFrame(class_metrics_records)
    class_metrics_df.to_csv('class_performance_detail.csv', index=False)
    print("Saved full per-fold class precision/recall detail to class_performance_detail.csv")

if all_y_test_pooled:
    pooled_y_test = np.concatenate(all_y_test_pooled)
    pooled_y_pred = np.concatenate(all_y_pred_pooled)
    print("\n--- Pooled Out-of-Sample Classification Report (all folds combined) ---")
    print(classification_report(
        pooled_y_test, pooled_y_pred,
        labels=class_labels_ordered,
        target_names=[CLASS_NAMES[l] for l in class_labels_ordered],
        zero_division=0
    ))
# ============================================================================
# STEP 9 & 10 — Advanced Trade Performance & Rolling Expectancy CVaR
# ============================================================================
print(f"\n--- Strategy Rolling Expectancy & Stability by {prob_label} Threshold ---")
print("-" * 140)
print(f"{'Thresh':<8} | {'Trades':<8} | {'Win%':<8} | {'RR':<6} | {'Overall Exp':<12} | {'Exp CVaR (5%)':<14} | {'Stable (CVaR>0)':<16} | {'Total R'}")
print("-" * 140)

ROLLING_WINDOW = 30  # Number of trades to define a "streak" or "period"

for t in thresholds_to_track:
    if len(trade_log[t]) == 0:
        continue
        
    log_df = pd.DataFrame(trade_log[t], columns=['datetime', 'ret'])
    returns = log_df['ret'].values
    num_trades = len(returns)
    
    # Standard metrics
    win_rate = np.mean(returns > 0)
    avg_win = returns[returns > 0].mean() if sum(returns > 0) > 0 else 0
    avg_loss = returns[returns < 0].mean() if sum(returns < 0) > 0 else 0
    rr = (avg_win / abs(avg_loss)) if avg_loss != 0 else np.nan
    
    # Overall Expectancy (Mean R per trade)
    overall_exp = np.mean(returns)
    total_r = np.sum(returns)
    
    # ---------------------------------------------------------
    # ROLLING EXPECTANCY CVaR
    # ---------------------------------------------------------
    if num_trades >= ROLLING_WINDOW:
        # 1. Calculate local expectancy across rolling streaks
        rolling_exp = pd.Series(returns).rolling(window=ROLLING_WINDOW).mean().dropna()
        
        # 2. Isolate the worst 5% of these rolling streaks
        alpha = 0.05
        cutoff_idx = max(1, int(len(rolling_exp) * alpha))
        worst_rolling_streaks = np.sort(rolling_exp.values)[:cutoff_idx]
        
        # 3. Calculate CVaR (Expected Shortfall of the tail)
        cvar_exp = np.mean(worst_rolling_streaks)
        
        # 4. Stability Check: Is the model still profitable in its worst 5% of streaks?
        stable_flag = "YES" if cvar_exp > 0 else "NO"
    else:
        cvar_exp = np.nan
        stable_flag = "N/A (low trades)"

    # String formatting
    print(f">= {t*100:>3.0f}% | {num_trades:<8} | {win_rate:>5.1%} | {rr:<6.2f} | {overall_exp:>10.2f} R | {cvar_exp:>12.2f} R | {stable_flag:<16} | {total_r:>+6.2f} R")

print("-" * 140)
print(f"Note: 'Exp CVaR (5%)' represents the strategy's average R per trade during its worst 5% of {ROLLING_WINDOW}-trade streaks.")

# ============================================================================
# STEP 11 — Out-of-Sample PnL Chart & Trade Log Export (INTERACTIVE)
# ============================================================================
import plotly.graph_objects as go
import pandas as pd

print("\n--- Saving Trade Logs to CSV ---")

all_trades_list = []
fig = go.Figure()

for i, t in enumerate(thresholds_to_track):
    if len(trade_log[t]) == 0:
        continue

    log_df = pd.DataFrame(trade_log[t], columns=['datetime', 'ret']).sort_values('datetime')
    log_df['threshold'] = round(t, 2)
    log_df['cum_pnl'] = log_df['ret'].cumsum()
    
    all_trades_list.append(log_df)
    trade_numbers = list(range(1, len(log_df) + 1))
    
    fig.add_trace(go.Scatter(
        x=trade_numbers, 
        y=log_df['cum_pnl'], 
        mode='lines', 
        name=f'>= {t:.2f}',
        hovertemplate='Trade: %{x}<br>PnL: %{y:.2f}R<extra></extra>'
    ))

if all_trades_list:
    master_trade_df = pd.concat(all_trades_list)
    master_trade_df.to_csv('walk_forward_trade_log.csv', index=False)
    print("Full trade log saved to: walk_forward_trade_log.csv")

fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)

chart_title = f'Out-of-Sample PnL — Target: {target_name} | Inverted: {INVERT_RETURNS}'
fig.update_layout(
    title=chart_title,
    xaxis_title='Trade Number',
    yaxis_title='Cumulative Return (R)',
    legend_title=f'{prob_label} ≥ (Click!)',
    hovermode="x unified",
    template="plotly_white"
)

fig.write_html('pnl_walkforward_interactive.html')
print("Interactive chart saved to: pnl_walkforward_interactive.html")
fig.show()

# ============================================================================
# STEP 12 — Summary Report
# ============================================================================
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Sequential (order-aware) binary barrier labels, full dataset: {label_counts.to_dict()}")
print(f"Hunting zone ({RSI_FILTER_COLUMN} < {RSI_FILTER_THRESHOLD}) rows: {len(df_training_zone)}")
print(f"Trading Setup: Target {target_name} | Inverting Trade Direction: {INVERT_RETURNS}")

for lbl in class_labels_ordered:
    p_arr = np.array(class_metrics_by_label[lbl]['precision'])
    if len(p_arr) > 0:
        print(f"  {CLASS_NAMES[lbl]:<8} class precision (argmax): {p_arr.mean():.2%} ± {p_arr.std():.2%} across folds")

best_rr_threshold, best_rr = None, -np.inf
for t in thresholds_to_track:
    rr_arr = np.array([r for r in fold_metrics[t]['rrs'] if not np.isnan(r)])
    if len(rr_arr) > 0 and rr_arr.mean() > best_rr:
        best_rr, best_rr_threshold = rr_arr.mean(), t

if best_rr_threshold is not None:
    print(f"Best mean out-of-sample RR across folds: {best_rr:.2f} at {prob_label} >= {best_rr_threshold:.2f}")

for t in thresholds_to_track:
    if len(trade_log[t]) > 0:
        log_df = pd.DataFrame(trade_log[t], columns=['datetime', 'ret']).sort_values('datetime')
        cum = log_df['ret'].cumsum().iloc[-1]
        print(f"  {prob_label} >= {t:.2f}: {len(log_df)} trades, cumulative {cum:.2f}R out-of-sample")

print("\nReminder: this run ignores fees, slippage, latency, and partial fills.")
print("A positive cumulative R here is a necessary but not sufficient condition")
print("for live profitability — re-run with realistic transaction costs before sizing capital.")
print("=" * 70)

Label distribution (full dataset): {'LOSS': 238096, 'WIN': 43436}
Total rows in hunting zone: 223972
Label distribution (hunting zone): {'LOSS': 212437, 'WIN': 11535}

STOP BARRIER METRICS
Average Stop Barrier Size (Return): 0.000053
Average Stop Barrier Size (%):      0.0053%
Min Stop Barrier Size (%):          0.0000%
Max Stop Barrier Size (%):          1.0612%

--- Starting 15 Walk-Forward Folds | Targeting: WIN | Inverting Trades: True ---


c:\Users\Himaska\Documents\Projects\Business_Projects\intraday-momentum-composite\venv\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Fold 01 | Train Rows: 78060  | Tested on forward 10000 rows. Purged: 330


c:\Users\Himaska\Documents\Projects\Business_Projects\intraday-momentum-composite\venv\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Fold 02 | Skipped PFI — y_test lacks both classes (cannot compute ROC-AUC).
Fold 02 | Train Rows: 86828  | Tested on forward 10000 rows. Purged: 600


...

c:\Users\Himaska\Documents\Projects\Business_Projects\intraday-momentum-composite\venv\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Fold 14 | Skipped PFI — y_test lacks both classes (cannot compute ROC-AUC).
Fold 14 | Train Rows: 195284 | Tested on forward 10000 rows. Purged: 600


c:\Users\Himaska\Documents\Projects\Business_Projects\intraday-momentum-composite\venv\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Fold 15 | Skipped PFI — y_test lacks both classes (cannot compute ROC-AUC).
Fold 15 | Train Rows: 204322 | Tested on forward 10000 rows. Purged: 600

--- Feature Importance: SHAP vs Permutation (PFI) ---
--------------------------------------------------------------------------------
        Feature  SHAP_Importance  PFI_Mean_AUC_Drop
execution_price         1.029571       1.235485e-01
          rsi_2         1.513403       4.720917e-02
    candle_type         0.164342       4.089495e-02
  largest_buyer         0.212345       2.218870e-02
    candle_size         0.136048       4.744088e-03
         volume         0.402447       1.269672e-03
      micro_CVD         0.015165       1.138523e-03
        buy_vol         0.000394      -1.837254e-07
      micro_OFI         0.002013      -1.554010e-06
       sell_vol         0.000340      -9.729790e-06
          CVD_z         0.146075      -1.285565e-03
        OFI_AGG         0.080622      -2.110813e-03
         sma_14         0.195923      -


SUMMARY
Sequential (order-aware) binary barrier labels, full dataset: {'LOSS': 238096, 'WIN': 43436}
Hunting zone (rsi_2 < 30) rows: 223972
Trading Setup: Target WIN | Inverting Trade Direction: True
  LOSS     class precision (argmax): 99.20% ± 3.01% across folds
  WIN      class precision (argmax): 3.47% ± 12.97% across folds
Best mean out-of-sample RR across folds: 1.23 at P(WIN) >= 0.55
  P(WIN) >= 0.50: 17 trades, cumulative 5.73R out-of-sample
  P(WIN) >= 0.55: 15 trades, cumulative 3.98R out-of-sample
  P(WIN) >= 0.60: 13 trades, cumulative 3.73R out-of-sample
  P(WIN) >= 0.65: 12 trades, cumulative 2.22R out-of-sample
  P(WIN) >= 0.70: 11 trades, cumulative 1.22R out-of-sample
  P(WIN) >= 0.75: 10 trades, cumulative 2.22R out-of-sample
  P(WIN) >= 0.80: 8 trades, cumulative 1.71R out-of-sample
  P(WIN) >= 0.85: 7 trades, cumulative 1.05R out-of-sample
  P(WIN) >= 0.90: 4 trades, cumulative 2.00R out-of-sample
  P(WIN) >= 0.95: 3 trades, cumulative 1.00R out-of-sample

Reminder

# Conclusion (10k samples split, 15 folds)
![Performance](./image/img_15.png)
![Performance](./image/img_16.png)
![Performance](./image/img_17.png)

With condition 1.5RR, timeout 240s, and barrier multiplier set as 1.0 it shows potential for alpha signal where RSI(2) and orderflow shows a statistical information with each other using triple barrier method.

# Possible Flaws with the Research

## RR VaR (Timeouts and Barriers)
The metric allows for the model to benefit from truncation of loss distributions. This is a false 'RR' where it is explained as the following.

$$RR_{real} = \frac{\mathbb{E}[W]}{\mathbb{E}[L_{SL} \cup L_{TO}]} = \frac{PT}{\frac{\sum_{i \in L_{SL}} \vert{}SL_i\vert{} + \sum_{j \in L_{TO}} \vert{}TO_j\vert{}}{\vert{}L_{SL}\vert{} + \vert{}L_{TO}\vert{}}}$$

- Let fixed PT = 1.5 and SL 1.0. Then $TO \in (-1.0, 1.5)$, and in practice $TO \approx 0$.
- As the ratio of $\vert{}L_{TO}\vert{}$ increases relative to $\vert{}L_{SL}\vert{}$, the denominator $\mathbb{E}[L]$ shrinks significantly.
- Even if your "Win Rate" is only 52%, the skewness introduced by the "Timeout" losses forces the denominator toward zero, mathematically inflating the $RR$ to 2.5 or 3.0.
- I treat my strategy as a stochastic process with absorbing barriers. When I initially evaluated my Risk/Reward (RR) profile, I observed that it appeared abnormally high. I realized this wasn't an alpha signal, but a statistical artifact: my returns are not Normally distributed; they are subject to compact support (defined by the $-1R$ and $+1.5R$ limits) and Type I censoring (via the timeout condition).Using standard deviation as a risk metric was inappropriate here because it assumes infinite tails, which significantly overestimates risk in a bounded system. Instead, I’ve moved toward a regime-based analysis, partitioning my performance into 'Barrier-Hit' and 'Timeout' classes to isolate true price prediction from noise.

## Model Lookahead Bias
Recall and precision shows model is exceptionally good in predicting losses (0.95, 0.96) despite having balanced weights. This might be a possible information leakage as achieving this kind of precision is unrealistic in the market (Wrong definition on triple barrier method or purge split data issue causing leakage). Loss binary classification should be treated as a Type I error.

# Closing
Overall there might be a relationship between RSI(2) snapback and complex orderflow patterns and features. I acknowledge that StDev assumes normality which is violated by the bounded nature of triple barrier method. The research currently lacks formal hypothesis testing and future iterations will be implemented to confirm alpha is not the result of stochastic noise.